In [ ]:
import pandas as pd
from typing import Iterable
from helpers import get_request

In [ ]:
def get_article_contents(links: Iterable[str]):
    get_url_from_link = lambda link : f"https://www.ufc.com/{link}"

    link_len = len(links)

    for (i, link) in enumerate(links):
        if i % 10 == 0:
            print("Getting page", i, "/", link_len)
    
        try:
            response = get_request(get_url_from_link(link))
            yield response.text
        except Exception as e:
            print("Error getting page", link)
            print(e)

            yield None

In [ ]:
filename = "./article_links_16-09-25_12-21-53"
articles_df = pd.read_csv(f"{filename}.csv")
articles_df

In [ ]:
articles_with_correct_type = articles_df[
    ((articles_df["type"] == "Results")
    | (articles_df["type"] == "Athletes")
    | (articles_df["type"] == "Fight Coverage")
    | (articles_df["type"] == "Fight Preview"))
    & (~articles_df["link"].str.contains("video"))
]
articles_with_correct_type

In [ ]:
articles_to_scrape = articles_with_correct_type[articles_with_correct_type["link"].str.contains("news")].copy()
articles_to_scrape

In [ ]:
contents = [page for page in get_article_contents(articles_to_scrape["link"])]
print("Got", len([page for page in contents if page is not None]), "page contents")

In [ ]:
articles_to_scrape["content"] = contents
articles_to_scrape

In [ ]:
articles_to_scrape\
    .set_index("link")\
    .to_csv(f"{filename}_with_contents.csv")